In [1]:
# --- Install training deps (run once), then verify imports ---
import sys, subprocess, pkgutil

def pip_install(pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", *pkgs])

# Core libs for QLoRA training
needed = [
    "transformers>=4.41.0",
    "peft>=0.11.1",
    "accelerate>=0.33.0",
    "datasets",
    "bitsandbytes",            # 4-bit quant (NVIDIA); will be ignored if no CUDA
    "sentence-transformers",
    "scikit-learn"
]
# Install any that are missing
missing = [p for p in ["transformers","peft","accelerate","datasets","bitsandbytes","sentence_transformers","sklearn"]
           if not pkgutil.find_loader(p)]
if missing:
    print("Installing:", missing)
    pip_install(needed)

# If you don't have PyTorch yet (or want CUDA), install a matching wheel.
# Uncomment ONE of the following lines that fits your setup:
# subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "torch", "--index-url", "https://download.pytorch.org/whl/cu121"])  # CUDA 12.1
# subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "torch==2.4.*"])  # CPU-only (generic)

# Verify imports and CUDA
import torch, transformers, peft, accelerate, datasets, bitsandbytes as bnb
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("accelerate:", accelerate.__version__)


/tmp/ipykernel_1177628/2148660271.py:19: DeprecationWarning: 'pkgutil.find_loader' is deprecated and slated for removal in Python 3.14; use importlib.util.find_spec() instead
  if not pkgutil.find_loader(p)]


torch: 2.7.1+cu126 | CUDA available: True
transformers: 4.57.1
peft: 0.17.1
accelerate: 1.10.1


In [2]:
!cp ./seinfeld_plots.jsonl train.jsonl


In [2]:
# premod_and_ner.py
# Clean JSONL -> pre-modernize -> NER -> write JSONL for inspection and IO pairs for training.

import os, json, re, hashlib, argparse, sys, pathlib, gc
from typing import List, Dict, Any, Tuple

# ----------------- config (overridable via env) -----------------
IN_PATH = os.getenv("IN_PATH", "./seinfeld_plots.jsonl")
OUT_JSONL = os.getenv("OUT_JSONL", "./seinfeld_plots.premod+ner.jsonl")
OUT_IO = os.getenv("OUT_IO", "./seinfeld_plots.io.train.jsonl")

ERA_GUIDE = os.getenv("ERA_GUIDE", "2025")
PLOT_STOP_MARK = os.getenv("PLOT_STOP_MARK", "### Plot End")
GEN_TEMP = float(os.getenv("GEN_TEMP", "0.8"))
GEN_TOP_P = float(os.getenv("GEN_TOP_P", "0.92"))
MAX_LEN = int(os.getenv("MAX_LEN", "4096"))
PREMOD_MAX_NEW = int(os.getenv("PREMOD_MAX_NEW", "320"))
MAX_NER_TOKENS = int(os.getenv("MAX_NER_TOKENS", "256"))  # bumped for richer JSON

# --- enforce Qwen 14B unless explicitly changed to another Qwen 14B variant
_QWEN_14B_DEFAULT = "Qwen/Qwen2.5-14B-Instruct"
BASE_MODEL = os.getenv("BASE_MODEL", _QWEN_14B_DEFAULT)
PREMOD_MODEL = os.getenv("PREMOD_MODEL", "")  # if blank, use BASE_MODEL
NER_MODEL = os.getenv("NER_MODEL", "")        # if blank, use BASE_MODEL
HF_TOKEN = (
    os.getenv("HF_TOKEN")
    or os.getenv("HUGGINGFACE_TOKEN")
    or os.getenv("HUGGINGFACEHUB_API_TOKEN")
    or os.getenv("HUGGING_FACE_HUB_TOKEN")
)

def _ensure_qwen14b(repo: str) -> str:
    # Safety: keep it on a Qwen 14B instruct checkpoint unless user passes another Qwen 14B explicitly
    r = (repo or "").strip()
    if not r or ("Qwen" not in r) or ("14B" not in r and "14b" not in r):
        return _QWEN_14B_DEFAULT
    return r

# ----------------- deps & HF -----------------
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig

def _ensure_pad(tok):
    if tok.pad_token is None:
        if tok.eos_token is not None: tok.pad_token = tok.eos_token
        else: tok.add_special_tokens({"pad_token": "[PAD]"})
    return tok

def _load_llm(repo: str):
    bnb_cfg = None
    try:
        import bitsandbytes as bnb  # noqa
        if torch.cuda.is_available():
            from transformers import BitsAndBytesConfig
            bnb_cfg = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.bfloat16
            )
    except Exception:
        pass
    tok = AutoTokenizer.from_pretrained(repo, use_fast=True, token=HF_TOKEN, trust_remote_code=True)
    _ensure_pad(tok)
    model = AutoModelForCausalLM.from_pretrained(
        repo, token=HF_TOKEN, trust_remote_code=True,
        quantization_config=bnb_cfg,
        dtype=(torch.bfloat16 if (bnb_cfg and torch.cuda.is_available()) else torch.float32),
        device_map="auto"
    )
    model.eval()
    return tok, model

# ----------------- utility -----------------
CONTROL_FIX = {
    "�": "-",
    "\x13": "-",
    "\x14": "-",
    "": "-",
    "": "-",
}
def clean(s: str) -> str:
    if not isinstance(s, str): return s
    for k, v in CONTROL_FIX.items():
        s = s.replace(k, v)
    return s

def hash_text(s: str) -> str:
    return hashlib.md5(clean(s.strip()).encode("utf-8")).hexdigest()

def apply_sampling_config(model, temperature, top_p):
    # set via generation_config to avoid "ignored" warnings
    try:
        cfg = GenerationConfig.from_model_config(model.config)
    except Exception:
        cfg = model.generation_config
    for k, v in (("do_sample", True), ("temperature", float(temperature)), ("top_p", float(top_p))):
        if hasattr(cfg, k): setattr(cfg, k, v)
    # avoid top_k to silence warnings
    if hasattr(cfg, "top_k"): cfg.top_k = None
    model.generation_config = cfg

class Stopper:
    # simple substring trim (reliable across tokenizers)
    @staticmethod
    def trim_at_stop(s: str, stop: str) -> str:
        return s.split(stop, 1)[0].rstrip() if stop in s else s

def _json_block(s: str) -> str:
    """Extract the first plausible JSON object; lightly repair common mistakes."""
    m = re.search(r"\{.*\}", s, flags=re.S)
    if not m:
        return "{}"
    payload = m.group(0).strip()
    # fix trailing commas and single quotes in a conservative way
    payload = re.sub(r",\s*}", "}", payload)
    payload = re.sub(r",\s*]", "]", payload)
    # If model used single quotes, try to convert only for keys/strings
    if "'" in payload and '"' not in payload.split("{",1)[1]:
        payload = payload.replace("'", '"')
    return payload

def _dedup_list(vals: List[str]) -> List[str]:
    seen = set(); out = []
    for v in vals:
        t = str(v).strip()
        if not t: continue
        k = re.sub(r"\s+", " ", t).lower()
        if k in seen: continue
        seen.add(k); out.append(t)
    return out

# ----------------- pre-modernization -----------------
STYLE_CARD = f"""You are a sitcom plot writer updating legacy episode *plots* to feel current in {ERA_GUIDE}.
Keep core character dynamics and comedic beats, but:
- Update references (tech, social platforms, slang, brands, news) to feel {ERA_GUIDE}.
- Swap outdated devices with modern equivalents (DMs, group chats, ride-hailing, wireless earbuds).
- Use contemporary New York details (contactless payments, delivery apps, e-bikes, remote/hybrid work).
- Avoid anachronisms. Prefer 2023-{ERA_GUIDE} references.
- Keep it PG-13, fast banter, short beats, crisp callbacks.
"""

def premodernize_prompt(stop_mark: str) -> str:
    return (
        clean(STYLE_CARD)
        + "\nTask: Rewrite the following *plot outline* to feel current in "
        + f"{ERA_GUIDE}. Keep characters/dynamics and comedic beats; DO NOT include production trivia, air dates, "
          "season/episode numbers, network names (e.g., NBC), or encyclopedic statements like X is the 147th episode&. "
          "Write as a compact outline (4-8 beats), no dialogue, no scene headers. "
        + f"End with the exact line: {stop_mark}\n"
    )

TRIVIA_RE = re.compile(r"(?i)\b(episode|season|aired|runtime|network|NBC|CBS|ABC|FOX|Wiki|IMDb|production|No\.\s?\d+|air date|viewers|ratings)\b")
YEAR_RE = re.compile(r"\b(19|20)\d{2}\b")

def strip_trivia(text: str) -> str:
    kept = []
    for l in text.splitlines():
        l0 = clean(l)
        if TRIVIA_RE.search(l0) or YEAR_RE.search(l0):
            if not re.search(r"(?i)\b(Jerry|Elaine|George|Kramer|Newman|Monk|apartment|date|party|argument|scheme|mistake)\b", l0):
                continue
        kept.append(l0)
    return "\n".join(kept).strip()

def premodernize(text: str, tok, model) -> str:
    guide = premodernize_prompt(PLOT_STOP_MARK)
    text = strip_trivia(clean(text))
    if hasattr(tok, "apply_chat_template"):
        prompt = tok.apply_chat_template(
            [{"role":"system","content":guide},
             {"role":"user","content":f'Original plot:\n\"\"\"\n{text}\n\"\"\"'}],
            tokenize=False, add_generation_prompt=True
        )
    else:
        prompt = guide + "\nOriginal plot:\n\"\"\"\n" + text + "\n\"\"\"\nAssistant:"
    enc = tok(prompt, return_tensors="pt", truncation=True, max_length=min(4096, MAX_LEN))
    enc = {k:v.to(model.device) for k,v in enc.items()}
    apply_sampling_config(model, GEN_TEMP, GEN_TOP_P)
    with torch.no_grad():
        out = model.generate(
            **enc,
            max_new_tokens=PREMOD_MAX_NEW,
            eos_token_id=None,
            pad_token_id=tok.pad_token_id,
        )
    gen = tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    gen = Stopper.trim_at_stop(gen, PLOT_STOP_MARK)
    return clean(gen)

# ----------------- NER -----------------
# Rich schema covering "all valuable NEs"
NER_KEYS = [
    "characters", "places", "organizations",
    "foods", "clothing", "objects", "furniture", "rooms",
    "vehicles", "weather", "digital",
    "animals", "events", "media", "routes", "amounts", "time",
    "other"
]

def run_ner(plot_text: str, tok, model, max_new: int = MAX_NER_TOKENS) -> Dict[str, List[str]]:
    sysmsg = (
        "You are a high-precision information extraction model for sitcom plots. "
        "Extract salient, concrete named entities and terms from the plot and categorize them strictly into the provided keys. "
        "Prefer exact surface forms from the text; do NOT infer new brands or names."
    )
    # Tight JSON spec to guide Qwen into the right shape
    schema_hint = {
        "characters": ["Jerry", "Elaine"],
        "places": ["Monk's Cafe", "apartment lobby"],
        "organizations": ["Yankees"],
        "foods": ["latte", "bagel"],
        "clothing": ["puffy jacket"],
        "objects": ["key fob", "phone"],
        "furniture": ["couch"],
        "rooms": ["kitchen"],
        "vehicles": ["taxi", "e-bike", "Subway N line"],
        "weather": ["rain"],
        "digital": ["DM", "group chat", "ride-hailing app"],
        "animals": [],
        "events": ["playoff game"],
        "media": ["livestream"],
        "routes": ["JFK Airport"],
        "amounts": ["$20"],
        "time": ["morning"],
        "other": []
    }

    rules = (
        "Guidelines:\n"
        "- characters: person names as written (first/last/nicknames).\n"
        "- places: venues/shops/restaurants/cafes/bars/parks/stations, apartments, streets, neighborhoods.\n"
        "- organizations: companies, teams, media outlets, agencies.\n"
        "- foods: dishes/snacks/drinks.\n"
        "- clothing: garments/accessories (e.g., hoodie, blazer, scarf).\n"
        "- objects: tangible items (e.g., phone, receipt, coffee machine).\n"
        "- furniture: couch, table, stool, desk.\n"
        "- rooms: kitchen, bathroom, hallway, office.\n"
        "- vehicles: cars, taxis, rideshares, scooters, e-bikes, subway/bus lines.\n"
        "- weather: rain, snow, heatwave, humidity, blizzard.\n"
        "- digital: apps, platforms, messages, notifications, livestreams.\n"
        "- animals: pets/animals named.\n"
        "- events: games, parties, meetings, ceremonies.\n"
        "- media: podcasts, streams, TV shows (as mentioned in-plot).\n"
        "- routes: airports, lines, stations, intersections.\n"
        "- amounts: explicit money/quantities (e.g., $50, 2 tickets).\n"
        "- time: named times (e.g., morning, midnight, lunch rush).\n"
        "- other: anything salient not covered above.\n"
        "Rules: use surface forms, deduplicate near duplicates, keep each list d 40 items, drop generic words.\n"
        "Return STRICT JSON with exactly these keys and array-of-string values for each:\n"
        f"{list(schema_hint.keys())}\n"
        "No prose, no markdown."
    )

    user = (
        "Plot:\n\"\"\"\n" + plot_text.strip() + "\n\"\"\"\n\n"
        "Respond only with JSON."
    )

    if hasattr(tok, "apply_chat_template"):
        prompt = tok.apply_chat_template(
            [{"role":"system","content":sysmsg},
             {"role":"user","content":rules},
             {"role":"user","content":json.dumps({"schema_example": schema_hint}, ensure_ascii=False)},
             {"role":"user","content":user}],
            tokenize=False, add_generation_prompt=True
        )
    else:
        prompt = sysmsg + "\n\n" + rules + "\n\n" + json.dumps({"schema_example": schema_hint}, ensure_ascii=False) + "\n\n" + user + "\nAssistant:"

    enc = tok(prompt, return_tensors="pt", truncation=True, max_length=2048)
    enc = {k:v.to(model.device) for k,v in enc.items()}
    with torch.no_grad():
        out = model.generate(
            **enc, max_new_tokens=max_new, do_sample=False,
            eos_token_id=tok.eos_token_id, pad_token_id=tok.pad_token_id
        )
    gen = tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    payload = _json_block(gen)

    obj: Dict[str, Any]
    try:
        obj = json.loads(payload)
    except Exception:
        obj = {}

    # Normalize to full schema
    by_type: Dict[str, List[str]] = {}
    for k in NER_KEYS:
        raw = obj.get(k, [])
        if isinstance(raw, list):
            vals = [str(x) for x in raw if isinstance(x, (str, int, float)) or x]
        else:
            # Handle dicts or singletons
            if isinstance(raw, (str, int, float)):
                vals = [str(raw)]
            else:
                vals = []
        by_type[k] = _dedup_list(vals)[:40]

    # Fallback heuristics if characters/places empty
    if not by_type["characters"] or not by_type["places"]:
        caps = re.findall(r"\b[A-Z][a-z]+(?:\s+[A-Z][a-z]+)*\b", plot_text)
        if not by_type["characters"]:
            by_type["characters"] = _dedup_list(caps[:8])
        if not by_type["places"]:
            by_type["places"] = _dedup_list(caps[8:16])

    return by_type

# ----------------- IO prompt builder -----------------
def _flatten_for_io(entities_by_type: Dict[str, List[str]], max_total: int = 40) -> List[str]:
    # Prioritize for plot conditioning
    priority = [
        "characters", "places", "organizations",
        "foods", "objects", "vehicles", "clothing",
        "digital", "events", "rooms", "furniture",
        "weather", "routes", "animals", "amounts", "time", "media", "other"
    ]
    flat: List[str] = []
    for k in priority:
        flat.extend(entities_by_type.get(k, []))
        if len(flat) >= max_total: break
    # dedupe again while preserving order
    return _dedup_list(flat)[:max_total]

def build_io_prompt(entities_flat: List[str]) -> str:
    ent_str = "; ".join(entities_flat) if entities_flat else "Jerry; Elaine; George; Kramer"
    ent_str = clean(ent_str)
    return (
        f"Entities: {ent_str}\n"
        f"Task: Write a concise sitcom PLOT outline (not script) using ONLY these entities. "
        f"Keep 4-8 beats with cause-effect humor. Avoid dialogue.\n"
        f"Era: original 1990s tone; avoid anachronisms.\n"
        f"End with the exact line: {PLOT_STOP_MARK}"
    )

# ----------------- data loading / cleaning -----------------
def load_jsonl(path: str) -> List[Dict[str, Any]]:
    p = pathlib.Path(path)
    if not p.exists(): raise FileNotFoundError(path)
    rows, bad = [], 0
    with p.open("r", encoding="utf-8", errors="replace") as f:
        for i, line in enumerate(f, 1):
            s = line.strip()
            if not s: continue
            try:
                obj = json.loads(s)
            except Exception:
                bad += 1
                continue
            if isinstance(obj, dict) and any(k in obj for k in ("text","messages","input_text","output_text","title","plot_text")):
                rows.append(obj)
            else:
                bad += 1
    if bad:
        print(f"[cleaner] dropped {bad} malformed/unsupported lines", file=sys.stderr)
    return rows

def normalize_to_text(row: Dict[str, Any]) -> Tuple[str, str]:
    """returns (schema, text) where schema in {'text','title+plot','io','chat'}"""
    if "text" in row:
        return "text", clean(str(row["text"]))
    if "title" in row or "plot_text" in row:
        title = clean(str(row.get("title",""))).strip()
        plot = clean(str(row.get("plot_text",""))).strip()
        txt = (title + ("\n\n" if title and plot else "") + plot).strip()
        return "title+plot", txt
    if "input_text" in row and "output_text" in row:
        # treat output_text as the plot
        return "io", clean(str(row["output_text"]))
    if "messages" in row:
        try:
            txt = "\n".join([clean(m.get("content","")) for m in row["messages"]]).strip()
        except Exception:
            txt = ""
        return "chat", txt
    # fallback: serialize
    return "unknown", clean(json.dumps(row, ensure_ascii=False))

# ----------------- main pipeline -----------------
def main():
    print(f"[i] input: {IN_PATH}")
    base_repo = _ensure_qwen14b(BASE_MODEL)
    premod_repo = _ensure_qwen14b(PREMOD_MODEL or base_repo)
    ner_repo = _ensure_qwen14b(NER_MODEL or base_repo)

    print(f"[i] loading pre-modernizer: {premod_repo}")
    prem_tok, prem_model = _load_llm(premod_repo)

    print(f"[i] loading NER model: {ner_repo}")
    ner_tok, ner_model = _load_llm(ner_repo)

    rows = load_jsonl(IN_PATH)
    print(f"[i] loaded {len(rows)} lines")

    out_full = pathlib.Path(OUT_JSONL)
    out_full.parent.mkdir(parents=True, exist_ok=True)
    out_io = pathlib.Path(OUT_IO)
    out_io.parent.mkdir(parents=True, exist_ok=True)

    kept_full = kept_io = 0
    with out_full.open("w", encoding="utf-8") as wf, out_io.open("w", encoding="utf-8") as wi:
        for idx, r in enumerate(rows):
            print("idx: ", idx, end=" ** ")
            schema, original = normalize_to_text(r)
            if not original:
                continue

            # pre-modernize
            modern = premodernize(original, prem_tok, prem_model)
            # short sanity: if rewrite is too short, keep original
            if len(modern.split()) < 60:
                modern = original

            # NER on modern text
            try:
                entities_by_type = run_ner(modern, ner_tok, ner_model)
            except Exception:
                entities_by_type = {k: [] for k in NER_KEYS}

            entities_flat = _flatten_for_io(entities_by_type, max_total=40)

            record = {
                "schema": schema,
                "entities_by_type": entities_by_type,
                "entities_flat": entities_flat,
                "original": original,
                "modern_plot": modern,
            }
            wf.write(json.dumps(record, ensure_ascii=False) + "\n"); kept_full += 1

            # IO pair for training
            prompt = build_io_prompt(entities_flat)
            wi.write(json.dumps({
                "schema": "io",
                "input_text": prompt,
                "output_text": modern
            }, ensure_ascii=False) + "\n")
            kept_io += 1

    # free VRAM
    del prem_model, prem_tok, ner_model, ner_tok
    gc.collect()
    try:
        if torch.cuda.is_available():
            torch.cuda.empty_cache(); torch.cuda.ipc_collect()
    except Exception:
        pass

    print(f"[ok] wrote {kept_full} rows -> {out_full}")
    print(f"[ok] wrote {kept_io} IO rows -> {out_io}")
    print("[done]")

if __name__ == "__main__":
    main()

[i] input: ./seinfeld_plots.jsonl
[i] loading pre-modernizer: Qwen/Qwen2.5-14B-Instruct


Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

[i] loading NER model: Qwen/Qwen2.5-14B-Instruct


Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

[cleaner] dropped 1 malformed/unsupported lines


[i] loaded 171 lines
idx:  0 ** 

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


idx:  1 ** idx:  2 ** idx:  3 ** idx:  4 ** idx:  5 ** idx:  6 ** idx:  7 ** idx:  8 ** idx:  9 ** idx:  10 ** idx:  11 ** idx:  12 ** idx:  13 ** idx:  14 ** idx:  15 ** idx:  16 ** idx:  17 ** idx:  18 ** idx:  19 ** idx:  20 ** idx:  21 ** idx:  22 ** idx:  23 ** idx:  24 ** idx:  25 ** idx:  26 ** idx:  27 ** idx:  28 ** idx:  29 ** idx:  30 ** idx:  31 ** idx:  32 ** idx:  33 ** idx:  34 ** idx:  35 ** idx:  36 ** idx:  37 ** idx:  38 ** idx:  39 ** idx:  40 ** idx:  41 ** idx:  42 ** idx:  43 ** idx:  44 ** idx:  45 ** idx:  46 ** idx:  47 ** idx:  48 ** idx:  49 ** idx:  50 ** idx:  51 ** idx:  52 ** idx:  53 ** idx:  54 ** idx:  55 ** idx:  56 ** idx:  57 ** idx:  58 ** idx:  59 ** idx:  60 ** idx:  61 ** idx:  62 ** idx:  63 ** idx:  64 ** idx:  65 ** idx:  66 ** idx:  67 ** idx:  68 ** idx:  69 ** idx:  70 ** idx:  71 ** idx:  72 ** idx:  73 ** idx:  74 ** idx:  75 ** idx:  76 ** idx:  77 ** idx:  78 ** idx:  79 ** idx:  80 ** idx:  81 ** idx:  82 ** idx:  83 ** idx:  84 ** i

In [3]:
# kfold_local_sitcom_no_saves.py
# K-fold PEFT finetune for ENTITY->PLOT using precomputed entities + modern plots.
# Input  = original (90s) plot
# Target = "Entities: ...\nModernized plot:\n... \n### Plot End"
# Model  = Qwen/Qwen2.5-14B-Instruct

import os, sys, subprocess, importlib.util, json, random, gc, re, hashlib
from pathlib import Path
from datetime import datetime

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ----------------------- light auto-install -----------------------
if int(os.getenv("AUTO_PIP", "1")) == 1:
    def _need(pkg): return importlib.util.find_spec(pkg) is None
    to_install = []
    for mod, pipname in [
        ("transformers", "transformers>=4.41.0"),
        ("datasets", "datasets>=2.19.0"),
        ("peft", "peft>=0.11.1"),
        ("bitsandbytes", "bitsandbytes>=0.43.1"),
        ("pandas", "pandas>=2.2.0"),
        ("numpy", "numpy>=1.26.0"),
        ("torch", "torch>=2.2.0"),
        ("accelerate", "accelerate>=0.33.0"),
        ("tqdm", "tqdm>=4.66.0"),
        ("ipython", "ipython"),
        ("sentence_transformers", "sentence-transformers>=2.7.0"),
    ]:
        if _need(mod):
            to_install.append(pipname)
    if to_install:
        print("[setup] Installing:", to_install)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + to_install, check=False)

# -------------------- env knobs BEFORE HF imports -----------------
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")
os.environ.setdefault("WANDB_MODE", "disabled")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:512")
os.environ.setdefault("TQDM_DISABLE", "0")
os.environ.setdefault("TQDM_NOTEBOOK", "1")
os.environ.setdefault("PYTHONUNBUFFERED", "1")
os.environ.setdefault("DO_SAMPLE", "1")
os.environ.setdefault("GEN_TEMP", "0.8")
os.environ.setdefault("GEN_TOP_P", "0.92")

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoTokenizer

try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

CONTROL_FIX = {"�":"-","\x13":"-","\x14":"-","":"-","":"-"}
def _clean(s: str) -> str:
    if not isinstance(s, str): return s
    for k, v in CONTROL_FIX.items(): s = s.replace(k, v)
    return s

VERBOSE = int(os.getenv("VERBOSE", "1"))
def now():
    from datetime import datetime
    return datetime.now().isoformat(timespec="seconds")
def log(msg, level=1):
    if VERBOSE >= level: print(f"[{now()}] {msg}", flush=True)

# =========================
# Global config (via env)
# =========================
BASE_MODEL  = os.getenv("BASE_MODEL", "Qwen/Qwen2.5-14B-Instruct").strip()
OUT_DIR     = os.getenv("OUT_DIR", "outputs")
PEFT_METHOD = os.getenv("PEFT_METHOD", "adalora").lower()

K_FOLDS = int(os.getenv("K_FOLDS", "4"))
VALID_FRAC_FROM_TRAIN = float(os.getenv("VALID_FRAC_FROM_TRAIN", "0.1"))
MAX_LEN    = int(os.getenv("MAX_LEN", "4096"))
EPOCHS     = int(os.getenv("EPOCHS", "8"))
LR         = float(os.getenv("LR", "7e-5"))
SEED       = int(os.getenv("SEED", "1331"))

TRAIN_JSON = os.getenv("TRAIN_JSON", os.path.expanduser("./seinfeld_plots.premod+ner.jsonl"))
VAL_JSON   = os.getenv("VAL_JSON", "").strip()

ERA_GUIDE = os.getenv("ERA_GUIDE", "2025")
STYLE_CARD = os.getenv("STYLE_CARD", f"""
You are a sitcom plot writer updating legacy episode *plots* to feel current in {ERA_GUIDE}.
Keep core character dynamics and comedic beats, but:
- Update references (tech, social platforms, slang, brands, news) to feel {ERA_GUIDE}.
- Swap outdated devices with modern equivalents (DMs, group chats, ride-hailing, wireless earbuds).
- Use contemporary New York details (contactless payments, delivery apps, e-bikes, remote/hybrid work).
- Avoid anachronisms. Prefer 2023-{ERA_GUIDE} references.
- Keep it PG-13, fast banter, short beats, crisp callbacks.
""").strip()
STYLE_CARD = _clean(STYLE_CARD)
ERA_TAG = os.getenv("ERA_TAG", "<|era_2025|>")

GEN_MIN_NEW_TOKENS = int(os.getenv("GEN_MIN_NEW_TOKENS", "160"))
GEN_MAX_NEW_TOKENS = int(os.getenv("GEN_MAX_NEW_TOKENS", "320"))
GEN_TEMP  = float(os.getenv("GEN_TEMP", "0.8"))
GEN_TOP_P = float(os.getenv("GEN_TOP_P", "0.92"))
RETRY_IF_SHORT = int(os.getenv("RETRY_IF_SHORT", "1"))
PLOT_STOP_MARK = _clean(os.getenv("PLOT_STOP_MARK", "### Plot End"))
METRIC_SAMPLE = int(os.getenv("METRIC_SAMPLE", "43"))

HF_TOKEN = (
    os.getenv("HF_TOKEN")
    or os.getenv("HUGGINGFACE_TOKEN")
    or os.getenv("HUGGINGFACEHUB_API_TOKEN")
    or os.getenv("HUGGING_FACE_HUB_TOKEN")
)

Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# -------------------- tolerant loader --------------------
def _rows_from_hf_json(path):
    ds = load_dataset("json", data_files=path, split="train")
    rows = []
    cn = set(ds.column_names)

    # Our produced file (schema:title+plot + entities & modern text)
    if {"schema","original","modern_plot"}.issubset(cn) and ("entities_flat" in cn or "entities_by_type" in cn):
        for ex in ds:
            rows.append({
                "schema": "title+plot+ents",
                "original": f"{ex.get('original','')}",
                "modern_plot": f"{ex.get('modern_plot','')}",
                "entities_flat": list(ex.get("entities_flat", []) or []),
                "entities_by_type": ex.get("entities_by_type", {}) or {}
            })
        return rows

    # Fallbacks
    if {"input_text","output_text"}.issubset(cn):
        for ex in ds: rows.append({"schema":"io","input_text":f"{ex['input_text']}", "output_text":f"{ex['output_text']}"})
        return rows
    if "modern_plot" in cn:
        for ex in ds: rows.append({"schema":"premod","modern_plot":f"{ex.get('modern_plot','')}","entities":[]})
        return rows
    if "messages" in cn:
        for ex in ds: rows.append({"schema":"chat","messages":ex["messages"]})
        return rows
    if {"title","plot_text"}.issubset(cn):
        for ex in ds:
            combined=(f"{ex.get('title','')}\n\n{ex.get('plot_text','')}").strip()
            rows.append({"schema":"text","text":combined})
        return rows
    if "text" in cn:
        for ex in ds: rows.append({"schema":"text","text":f"{ex['text']}"})
        return rows

    raise ValueError("Unsupported columns: " + ", ".join(ds.column_names))

def _rows_from_manual_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            s = line.strip()
            if not s: continue
            try:
                obj = json.loads(s)
            except Exception:
                continue
            if not isinstance(obj, dict): continue

            # Our produced lines (schema:title+plot)
            if (obj.get("schema") == "title+plot") and ("original" in obj) and ("modern_plot" in obj) and ("entities_flat" in obj or "entities_by_type" in obj):
                rows.append({
                    "schema":"title+plot+ents",
                    "original": obj.get("original",""),
                    "modern_plot": obj.get("modern_plot",""),
                    "entities_flat": list(obj.get("entities_flat", []) or []),
                    "entities_by_type": obj.get("entities_by_type", {}) or {}
                })
                continue

            # Other fallbacks
            if "input_text" in obj and "output_text" in obj:
                rows.append({"schema":"io","input_text":obj["input_text"], "output_text":obj["output_text"]}); continue
            if "modern_plot" in obj:
                rows.append({"schema":"premod","modern_plot":obj.get("modern_plot",""),"entities":[]}); continue
            if "messages" in obj:
                rows.append({"schema":"chat","messages":obj["messages"]}); continue
            if "text" in obj:
                rows.append({"schema":"text","text":obj["text"]}); continue
            if "title" in obj or "plot_text" in obj:
                rows.append({"schema":"text","text": (obj.get("title","") + "\n\n" + obj.get("plot_text","")).strip()}); continue
            rows.append({"schema":"text","text": json.dumps(obj, ensure_ascii=False)})
    if not rows:
        raise ValueError(f"No valid records parsed from {path}")
    return rows

def detect_schema_to_text_rows(path):
    path = path.strip()
    if not path: return []
    p = Path(path)
    if not p.exists(): raise FileNotFoundError(f"{path} not found.")
    try:
        return _rows_from_hf_json(path)
    except Exception as e:
        log(f"[loader] HF datasets parse failed for {path}: {type(e).__name__}: {e}", level=2)
        return _rows_from_manual_jsonl(path)

def merge_train_val_to_pool(train_path=None, val_path=None):
    train_path = train_path or TRAIN_JSON
    val_path = (val_path or VAL_JSON or "").strip()
    assert Path(train_path).exists(), f"{train_path} not found."
    train_rows = detect_schema_to_text_rows(train_path)
    val_rows = []
    if val_path and Path(val_path).exists() and Path(val_path).is_file():
        val_rows = detect_schema_to_text_rows(val_path)
    pool = train_rows + val_rows
    random.shuffle(pool)
    log(f"Pool merged: {len(train_rows)} train + {len(val_rows)} val = {len(pool)} total rows")
    return pool

def kfold_indices(n, k=4, seed=42):
    idx = list(range(n)); random.Random(seed).shuffle(idx)
    sizes = [n // k + (1 if i < n % k else 0) for i in range(k)]
    cuts=[0]; [cuts.append(cuts[-1]+s) for s in sizes]
    return [idx[cuts[i]:cuts[i+1]] for i in range(k)]

def _ensure_pad_token(tok: AutoTokenizer):
    if tok.pad_token is None:
        if tok.eos_token is not None: tok.pad_token = tok.eos_token
        else: tok.add_special_tokens({'pad_token': '[PAD]'})
    return tok

CANDIDATE_MODELS = ["Qwen/Qwen2.5-14B-Instruct"]

def try_load_tokenizer(repo_id: str):
    try:
        tok = AutoTokenizer.from_pretrained(
            repo_id, use_fast=True, token=HF_TOKEN, trust_remote_code=True
        )
        _ensure_pad_token(tok)
        return True
    except Exception as e:
        log(f"[resolver] tokenizer failed for {repo_id}: {type(e).__name__}: {e}", level=2)
        return False

def resolve_base_model():
    if BASE_MODEL and try_load_tokenizer(BASE_MODEL):
        log(f"Using base model for PEFT + generation: {BASE_MODEL} (PEFT_METHOD={PEFT_METHOD})")
        return BASE_MODEL
    for rid in CANDIDATE_MODELS:
        if try_load_tokenizer(rid):
            log(f"Using base model for PEFT + generation: {rid} (PEFT_METHOD={PEFT_METHOD})")
            return rid
    raise RuntimeError("No working base model found. Provide HF token or adjust CANDIDATE_MODELS.")

# =========================
# Worker code (as a file)
# =========================
worker_code = r'''
import os, sys, json, random, gc, csv, traceback, inspect, math, re, hashlib
from pathlib import Path
import numpy as np, pandas as pd, torch
from datasets import load_dataset, Dataset
from transformers import (AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer,
                          DataCollatorForLanguageModeling, TrainerCallback, StoppingCriteria, StoppingCriteriaList)
from peft import (LoraConfig, AdaLoraConfig, get_peft_model, prepare_model_for_kbit_training)
from tqdm.auto import tqdm

CONTROL_FIX = {"�":"-","\x13":"-","\x14":"-","":"-","":"-"}
def _clean(s: str) -> str:
    if not isinstance(s, str): return s
    for k, v in CONTROL_FIX.items(): s = s.replace(k, v)
    return s

VERBOSE = int(os.getenv("VERBOSE", "1"))
def now():
    from datetime import datetime
    return datetime.now().isoformat(timespec="seconds")
def log(msg, level=1):
    if VERBOSE >= level: print(f"[{now()}] {msg}", flush=True)

def flush_cuda():
    try:
        gc.collect()
        if torch.cuda.is_available():
            try: torch.cuda.synchronize()
            except Exception: pass
            try:
                for dev in range(torch.cuda.device_count()):
                    with torch.cuda.device(dev):
                        torch.cuda.empty_cache()
            except Exception: pass
            try: torch.cuda.ipc_collect()
            except Exception: pass
    except Exception: pass

def _norm(s: str) -> str:
    if s is None: return ""
    s = _clean(str(s))
    s = re.sub(r"\s+", " ", s).strip()
    return s

def _hash_text(s:str)->str:
    s = _clean(s.strip())
    return hashlib.md5(s.encode("utf-8")).hexdigest()

# ------------- args -------------
args_path = sys.argv[1]
with open(args_path, "r", encoding="utf-8") as f: A = json.load(f)

fold_id      = A["fold_id"]; base_repo = A["base_repo"]; out_root = A["out_root"]
test_idx     = A["test_idx"]; train_json= A["train_json"]; val_json = A["val_json"]
config       = A["config"];   HF_TOKEN  = A.get("HF_TOKEN"); seed=A.get("seed", 42)

STYLE_CARD = _clean(A.get("STYLE_CARD", ""))
ERA_GUIDE  = A.get("ERA_GUIDE", "2025")
ERA_TAG    = A.get("ERA_TAG", "<|era_2025|>")

PEFT_METHOD     = config["PEFT_METHOD"]
MAX_LEN         = int(config["MAX_LEN"])
EPOCHS          = int(config["EPOCHS"])
LR              = float(config["LR"])
VALID_FRAC      = float(config["VALID_FRAC"])

GEN_MIN_NEW_TOKENS = int(os.getenv("GEN_MIN_NEW_TOKENS","160"))
GEN_MAX_NEW_TOKENS = int(os.getenv("GEN_MAX_NEW_TOKENS","320"))
GEN_TEMP           = float(os.getenv("GEN_TEMP","0.8"))
GEN_TOP_P          = float(os.getenv("GEN_TOP_P","0.92"))
PLOT_STOP_MARK     = _clean(os.getenv("PLOT_STOP_MARK", "### Plot End"))
METRIC_SAMPLE      = int(os.getenv("METRIC_SAMPLE","43"))

# ---------- IO formatting ----------
def _fmt_answer(ents_flat, modern, stop_mark):
    ents_flat = [x for x in (ents_flat or []) if _norm(x)]
    ents_line = "Entities: " + ("; ".join(ents_flat) if ents_flat else "")
    modern_line = "Modernized plot:\n" + (modern or "").strip()
    return _clean(f"{ents_line}\n{modern_line}\n{stop_mark}")

# ---------- tolerant loader ----------
def _ensure_pad_token(tok):
    if tok.pad_token is None:
        if tok.eos_token is not None: tok.pad_token = tok.eos_token
        else: tok.add_special_tokens({'pad_token': '[PAD]'})
    return tok

def _rows_from_hf_json(path):
    ds = load_dataset("json", data_files=path, split="train")
    rows = []
    cn = set(ds.column_names)

    if {"schema","original","modern_plot"}.issubset(cn) and ("entities_flat" in cn or "entities_by_type" in cn):
        for ex in ds:
            rows.append({
                "schema": "title+plot+ents",
                "original": "%s" % ex.get("original",""),
                "modern_plot": "%s" % ex.get("modern_plot",""),
                "entities_flat": list(ex.get("entities_flat", []) or []),
                "entities_by_type": ex.get("entities_by_type", {}) or {}
            })
        return rows

    if "messages" in cn:
        for ex in ds: rows.append({"schema":"chat","messages":ex["messages"]})
    elif "input_text" in cn and "output_text" in cn:
        for ex in ds: rows.append({"schema":"io","input_text":"%s"%ex["input_text"],"output_text":"%s"%ex["output_text"]})
    elif "modern_plot" in cn:
        for ex in ds: rows.append({"schema":"premod","modern_plot":"%s"%ex.get("modern_plot",""),"entities":[]})
    elif "text" in cn:
        for ex in ds: rows.append({"schema":"text","text":"%s"%ex["text"]})
    elif {"title","plot_text"}.issubset(cn):
        for ex in ds:
            combined = (f"{ex.get('title','')}\n\n{ex.get('plot_text','')}").strip()
            rows.append({"schema":"text","text":combined})
    else:
        raise ValueError(f"Unsupported schema in {path}")
    return rows

def _rows_from_manual_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            s=line.strip()
            if not s: continue
            try: obj=json.loads(s)
            except Exception: continue
            if not isinstance(obj, dict): continue

            if (obj.get("schema") == "title+plot") and ("original" in obj) and ("modern_plot" in obj) and ("entities_flat" in obj or "entities_by_type" in obj):
                rows.append({
                    "schema":"title+plot+ents",
                    "original": obj.get("original",""),
                    "modern_plot": obj.get("modern_plot",""),
                    "entities_flat": list(obj.get("entities_flat", []) or []),
                    "entities_by_type": obj.get("entities_by_type", {}) or {}
                }); continue

            if "input_text" in obj and "output_text" in obj:
                rows.append({"schema":"io","input_text":obj["input_text"], "output_text":obj["output_text"]}); continue
            if "modern_plot" in obj:
                rows.append({"schema":"premod","modern_plot":obj.get("modern_plot",""),"entities":[]}); continue
            if "messages" in obj:
                rows.append({"schema":"chat","messages":obj["messages"]}); continue
            if "text" in obj:
                rows.append({"schema":"text","text":obj["text"]}); continue
            if "title" in obj or "plot_text" in obj:
                rows.append({"schema":"text","text": (obj.get("title","") + "\n\n" + obj.get("plot_text","")).strip()}); continue
            rows.append({"schema":"text","text": json.dumps(obj, ensure_ascii=False)})
    if not rows: raise ValueError(f"No valid records parsed from {path}")
    return rows

def detect_schema_to_text_rows(path):
    if not path: return []
    p = Path(path)
    if not p.exists(): raise FileNotFoundError(f"{path} not found.")
    try: return _rows_from_hf_json(path)
    except Exception: return _rows_from_manual_jsonl(path)

def merge_train_val_to_pool(train_path, val_path):
    train_rows = detect_schema_to_text_rows(train_path)
    val_rows   = detect_schema_to_text_rows(val_path) if (val_path and Path(val_path).exists()) else []
    pool = train_rows + val_rows
    random.shuffle(pool)
    return pool

# ---------- TrainingArgs compat ----------
def _training_args_compat(base_kwargs):
    from transformers import TrainingArguments
    sig = inspect.signature(TrainingArguments.__init__)
    allowed = set(sig.parameters.keys())
    filtered = {k:v for k,v in base_kwargs.items() if k in allowed}
    try:
        return TrainingArguments(**filtered)
    except TypeError:
        for bad in ["logging_strategy","evaluation_strategy","disable_tqdm","dataloader_num_workers",
                    "dataloader_pin_memory","report_to","logging_first_step","warmup_ratio","lr_scheduler_type",
                    "remove_unused_columns"]:
            filtered.pop(bad, None)
        return TrainingArguments(**filtered)

def _make_training_args(has_val: bool, out_dir_fold: str):
    base_kwargs = dict(
        output_dir=os.path.join(out_dir_fold, "peft_tmp"),
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=16,
        learning_rate=LR,
        lr_scheduler_type="cosine",
        warmup_ratio=0.1,
        logging_steps=1,
        logging_first_step=True,
        logging_strategy="steps",
        evaluation_strategy=("epoch" if has_val else "no"),
        save_steps=0, save_total_limit=0,
        bf16=torch.cuda.is_available(),
        gradient_checkpointing=True,
        report_to="none",
        disable_tqdm=False,
        remove_unused_columns=False,
        dataloader_num_workers=0,
        dataloader_pin_memory=True,
    )
    return _training_args_compat(base_kwargs)

def _safe_adalora_or_lora(total_step: int):
    from peft import LoraConfig, AdaLoraConfig
    try:
        if total_step < 60: raise RuntimeError("adalora_too_small")
        tinit  = max(20, int(total_step * 0.10))
        tfinal = max(tinit + 10, int(total_step * 0.90))
        deltaT = max(5, (tfinal - tinit)//10)
        return AdaLoraConfig(
            init_r=8, target_r=32, beta1=0.85, beta2=0.85,
            tinit=tinit, tfinal=tfinal, deltaT=deltaT,
            lora_alpha=32, lora_dropout=0.05,
            total_step=total_step, task_type="CAUSAL_LM"
        )
    except Exception:
        return LoraConfig(
            r=32, lora_alpha=64, lora_dropout=0.05,
            target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
            bias="none", task_type="CAUSAL_LM"
        )

# ----------------- PROGRESS + CSV -----------------
class LiveTQDM(TrainerCallback):
    def __init__(self):
        self.bar = None; self.last_step = 0; self.last_loss = None
    def on_train_begin(self, args, state, control, **kwargs):
        total = state.max_steps if state.max_steps and state.max_steps > 0 else None
        self.bar = tqdm(total=total, desc=f"fold {fold_id} training", dynamic_ncols=True, leave=True, file=sys.stdout)
        self.last_step = 0
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs:
            self.last_loss = logs["loss"]
            print(f"[{now()}] fold={fold_id} TRAIN step={state.global_step} epoch={getattr(state,'epoch',None)} loss={self.last_loss}", flush=True)
    def on_step_end(self, args, state, control, **kwargs):
        if self.bar is not None:
            delta = state.global_step - self.last_step
            if delta > 0:
                self.bar.update(delta); self.last_step = state.global_step
                if self.last_loss is not None:
                    try: self.bar.set_postfix_str(f"loss={float(self.last_loss):.4f}")
                    except Exception: pass
    def on_evaluate(self, args, state, control, metrics, **kwargs):
        ev = metrics.get("eval_loss", None)
        if ev is not None:
            print(f"[{now()}] fold={fold_id} EVAL epoch={getattr(state,'epoch',None)} step={state.global_step} eval_loss={ev}", flush=True)
            try: self.bar.set_postfix_str(f"eval_loss={float(ev):.4f}")
            except Exception: pass
    def on_train_end(self, args, state, control, **kwargs):
        if self.bar is not None:
            self.bar.close(); self.bar = None

class EvalEveryEpoch(TrainerCallback):
    def __init__(self, do_eval_at_start=True): self.do_eval_at_start = do_eval_at_start
    def on_train_begin(self, args, state, control, **kwargs):
        if self.do_eval_at_start: control.should_evaluate = True
    def on_epoch_end(self, args, state, control, **kwargs): control.should_evaluate = True

class LossCSVLogger(TrainerCallback):
    def __init__(self, csv_path, fold_id):
        self.csv_path = csv_path; self.fold_id = fold_id; Path(csv_path).parent.mkdir(parents=True, exist_ok=True)
        if not os.path.exists(csv_path):
            with open(csv_path, "w", newline="", encoding="utf-8") as f:
                import csv as _csv
                _csv.writer(f).writerow(["timestamp","fold","phase","epoch","global_step","loss","eval_loss","learning_rate"])
    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs: return
        loss = logs.get("loss"); lr = logs.get("learning_rate"); epoch=logs.get("epoch")
        step = int(getattr(state, "global_step", 0))
        with open(self.csv_path, "a", newline="", encoding="utf-8") as f:
            import csv as _csv
            _csv.writer(f).writerow([now(), self.fold_id, "train", epoch, step, loss, None, lr])
    def on_evaluate(self, args, state, control, metrics, **kwargs):
        ev = metrics.get("eval_loss"); epoch=getattr(state,"epoch",None); step=int(getattr(state,"global_step",0))
        with open(self.csv_path, "a", newline="", encoding="utf-8") as f:
            import csv as _csv
            _csv.writer(f).writerow([now(), self.fold_id, "eval", epoch, step, None, ev, None])

# --------- simple metrics ---------
_word_re = re.compile(r"\w+")
def _tokens(text): return _word_re.findall(text.lower())
def _jaccard_unigram(a, b):
    A, B = set(_tokens(a)), set(_tokens(b))
    if not A and not B: return 1.0
    if not A or not B:  return 0.0
    inter = len(A & B); union = len(A | B)
    return (inter / union) if union else 0.0
def _distinct(texts, n=2):
    total = 0; uniq = set()
    for t in texts:
        toks = _tokens(t)
        if len(toks) < n: continue
        grams = [tuple(toks[i:i+n]) for i in range(len(toks)-n+1)]
        total += len(grams); uniq.update(grams)
    return (len(uniq) / total) if total else 0.0

# ---------- test generation prompt ----------
def _cap_line(min_toks, max_toks, stop_mark):
    return (f"Length window: write a complete plot between {min_toks} and {max_toks} model tokens. "
            f"End with the exact line: {stop_mark}")

def _prompt_from_row(tok, r):
    cap = _cap_line(int(os.getenv("GEN_MIN_NEW_TOKENS","160")), int(os.getenv("GEN_MAX_NEW_TOKENS","320")), os.getenv("PLOT_STOP_MARK","### Plot End"))
    system = f"{_clean(STYLE_CARD)}\nDirective: Produce a sitcom *plot outline* (not script). {ERA_TAG}\n{cap}"
    if r["schema"]=="io":
        user = r.get("input_text","").strip()
        modernize = (f"\nModernize to {ERA_GUIDE}. Conclude with {PLOT_STOP_MARK}.")
        return _clean(f"{system}\n\n{user}{modernize}")
    else:
        return _clean(f"{system}\n\nEntities: Jerry; Elaine; George; Kramer\nTask: Write a concise sitcom PLOT outline.\nConclude with {PLOT_STOP_MARK}.")

# ---------- stopper ----------
class StopOnSubsequence(StoppingCriteria):
    def __init__(self, stop_ids: torch.LongTensor):
        super().__init__()
        self.stop_ids = stop_ids
        self.L = stop_ids.shape[0]
    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor, **kwargs) -> bool:
        if input_ids.shape[0] != 1: return False
        seq = input_ids[0]
        if seq.shape[0] < self.L: return False
        if torch.equal(seq[-self.L:], self.stop_ids.to(seq.device)): return True
        return False

def _build_stoppers(tok, stop_text):
    ids = tok(stop_text, add_special_tokens=False, return_tensors="pt")["input_ids"][0]
    return StoppingCriteriaList([StopOnSubsequence(ids)])

def _generate_plot(prompt, model, tok,
                   min_new, max_new, temp, top_p, stop_mark, retry_if_short=True):
    device = model.device
    enc = tok(prompt, return_tensors="pt", truncation=True, max_length=min(4096, int(os.getenv("MAX_LEN","4096"))))
    enc = {k:v.to(device) for k,v in enc.items()}
    stoppers = _build_stoppers(tok, stop_mark)
    do_smpl = bool(int(os.getenv("DO_SAMPLE","1")))

    out = model.generate(
        **enc,
        eos_token_id=None,
        pad_token_id=tok.pad_token_id,
        min_new_tokens=min_new,
        max_new_tokens=max_new,
        do_sample=do_smpl,
        temperature=(temp if do_smpl else None),
        top_p=(top_p if do_smpl else None),
        num_beams=1,
        stopping_criteria=stoppers,
    )
    gen = tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    if stop_mark in gen: gen = gen.split(stop_mark, 1)[0].rstrip()
    return _clean(gen)

# ----------------- main worker -----------------
def main_worker():
    try:
        random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
        fold_dir=os.path.join(out_root, f"fold_{fold_id}"); Path(fold_dir).mkdir(parents=True, exist_ok=True)

        base_tok = AutoTokenizer.from_pretrained(base_repo, use_fast=True, token=HF_TOKEN, trust_remote_code=True)
        if base_tok.pad_token is None:
            base_tok.pad_token = base_tok.eos_token if base_tok.eos_token is not None else base_tok.add_special_tokens({'pad_token': '[PAD]'}) or base_tok.pad_token
        base_tok.padding_side = "left"

        pool_raw = merge_train_val_to_pool(train_json, val_json)

        # Build IO pairs (preferred path)
        pool_io=[]
        for r in tqdm(pool_raw, desc=f"fold {fold_id} IO-build", dynamic_ncols=True):
            if r["schema"] == "title+plot+ents":
                original = _clean((r.get("original","") or "").strip())
                modern   = _clean((r.get("modern_plot","") or "").strip())
                ents     = r.get("entities_flat") or []
                if not original or not modern: continue
                input_text  = original
                output_text = _fmt_answer(ents, modern, PLOT_STOP_MARK)
                pool_io.append({"schema":"io","input_text":input_text,"output_text":output_text})
                continue
            # Fallbacks just skip
            txt = _clean((r.get("text","") or r.get("modern_plot","") or r.get("input_text","") or "").strip())
            if not txt: continue
            pool_io.append({"schema":"io","input_text":txt,"output_text":_fmt_answer([], txt, PLOT_STOP_MARK)})

        test_rows=[pool_io[i] for i in test_idx]
        train_rows_all=[pool_io[i] for i in range(len(pool_io)) if i not in test_idx]
        log(f"[child {fold_id}] test={len(test_rows)} | train+val={len(train_rows_all)}")

        val_size=max(1, int(len(train_rows_all)*VALID_FRAC)) if len(train_rows_all)>10 else 0
        random.shuffle(train_rows_all)
        val_rows=train_rows_all[:val_size] if val_size>0 else []
        train_rows=train_rows_all[val_size:] if val_size>0 else train_rows_all

        # 5) Load base model for training (fresh)  single import used (no inner shadowing)
        bnb_cfg = None
        try:
            import bitsandbytes as bnb  # noqa
            if torch.cuda.is_available():
                from transformers import BitsAndBytesConfig
                bnb_cfg = BitsAndBytesConfig(
                    load_in_4bit=True,
                    bnb_4bit_use_double_quant=True,
                    bnb_4bit_quant_type="nf4",
                    bnb_4bit_compute_dtype=torch.bfloat16
                )
                log("[PEFT] Using 4-bit quantization.")
        except Exception: pass

        base_model = AutoModelForCausalLM.from_pretrained(
            base_repo, token=HF_TOKEN, trust_remote_code=True,
            quantization_config=bnb_cfg,
            dtype=(torch.bfloat16 if (bnb_cfg and torch.cuda.is_available()) else torch.float32),
            device_map="auto"
        )
        if getattr(base_model.config, "pad_token_id", None) is None:
            base_model.config.pad_token_id = base_tok.pad_token_id
        base_model.config.use_cache = False
        try: base_model = prepare_model_for_kbit_training(base_model)
        except Exception: pass

        def flat_io(r): return _clean(r.get("input_text","").strip() + "\n\n" + r.get("output_text","").strip())
        train_texts=[flat_io(r) for r in train_rows]
        val_texts  =[flat_io(r) for r in val_rows]

        per_device_bs=1; grad_accum=16; world_size=int(os.environ.get("WORLD_SIZE","1"))
        eff_batch=per_device_bs*grad_accum*world_size
        updates_per_epoch=max(1, math.ceil(len(train_texts)/eff_batch))
        total_step=max(1, updates_per_epoch*EPOCHS)
        log(f"[child {fold_id}] total_step={total_step}; updates_per_epoch={updates_per_epoch}")

        peft_cfg=_safe_adalora_or_lora(total_step)
        peft_model=get_peft_model(base_model, peft_cfg)

        def tok_train(batch): return base_tok(batch["text"], truncation=True, max_length=MAX_LEN)
        def tok_val(batch):   return base_tok(batch["text"], truncation=True, max_length=min(MAX_LEN, 4096))

        train_ds = Dataset.from_list([{"text": t} for t in train_texts])
        val_ds   = Dataset.from_list([{"text": t} for t in (val_texts or [])]) if val_texts else None

        train_tok = train_ds.map(tok_train, batched=True, remove_columns=train_ds.column_names)
        val_tok   = (val_ds.map(tok_val, batched=True, remove_columns=val_ds.column_names) if val_ds else None)

        has_val = val_tok is not None and len(val_tok)>0
        args=_make_training_args(has_val, fold_dir)
        collator = DataCollatorForLanguageModeling(base_tok, mlm=False, pad_to_multiple_of=8)

        callbacks = [LossCSVLogger(os.path.join(fold_dir, "train_eval_logs.csv"), fold_id),
                     LiveTQDM(),
                     EvalEveryEpoch(do_eval_at_start=bool(has_val))]

        trainer = Trainer(
            model=peft_model,
            args=args,
            data_collator=collator,
            train_dataset=train_tok,
            eval_dataset=val_tok if has_val else None,
            callbacks=callbacks,
        )

        log(f"[child {fold_id}] Starting training (IO: original -> entities+modern).")
        trainer.train()

        final_eval_loss = None; best_eval_loss = None
        if has_val:
            eval_out = trainer.evaluate()
            final_eval_loss = float(eval_out.get('eval_loss'))
            log(f"[child {fold_id}] FINAL EVAL: eval_loss={final_eval_loss}")
            try:
                df = pd.read_csv(os.path.join(fold_dir, "train_eval_logs.csv"))
                ev = df[df["phase"]=="eval"]["eval_loss"].dropna().astype(float)
                if len(ev)>0: best_eval_loss = float(ev.min())
            except Exception: pass

        metrics_path   = os.path.join(fold_dir, "metrics.csv")
        samples_jsonl  = os.path.join(fold_dir, "test_samples.jsonl")
        samples_csv    = os.path.join(fold_dir, "test_samples.csv")

        try:
            rng = random.Random(1234 + fold_id)
            sample_idx = rng.sample(range(len(test_rows)), k=min(METRIC_SAMPLE, len(test_rows))) if len(test_rows) > 0 else []
            sample     = [test_rows[i] for i in sample_idx] if sample_idx else []
            prompts    = [_prompt_from_row(base_tok, r) for r in sample]

            gen_texts = []
            peft_model.eval()
            for p in prompts:
                if not p: gen_texts.append(""); continue
                gen = _generate_plot(p, peft_model, base_tok, GEN_MIN_NEW_TOKENS, GEN_MAX_NEW_TOKENS, GEN_TEMP, GEN_TOP_P, PLOT_STOP_MARK, True)
                gen_texts.append(gen)

            with open(samples_jsonl, "w", encoding="utf-8") as fj, open(samples_csv, "w", newline="", encoding="utf-8") as fc:
                w = csv.writer(fc); w.writerow(["fold","pool_index","schema","prompt","generation"])
                for i, pool_i in enumerate(sample_idx):
                    rec = {"fold": int(fold_id), "pool_index": int(pool_i), "schema": "io",
                           "prompt": prompts[i], "generation": gen_texts[i]}
                    fj.write(json.dumps(rec, ensure_ascii=False) + "\n")
                    w.writerow([fold_id, pool_i, "io", prompts[i], gen_texts[i]])

            continuity = float(np.mean([len(re.findall(r"\w+", t.lower())) for t in gen_texts])) if gen_texts else 0.0
            def _jacc(a,b):
                A,B=set(_tokens(a)), set(_tokens(b))
                if not A and not B: return 1.0
                if not A or not B: return 0.0
                return len(A&B)/len(A|B)
            coh_j = float(np.mean([_jacc(prompts[i], gen_texts[i]) for i in range(len(gen_texts))])) if gen_texts else 0.0

            from sentence_transformers import SentenceTransformer
            model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
            pe = model.encode(prompts, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False) if prompts else np.zeros((0,384))
            ge = model.encode(gen_texts, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False) if gen_texts else np.zeros((0,384))
            coh_sem = float((pe*ge).sum(axis=1).mean()) if len(gen_texts)>0 else 0.0

            distinct2 = float(_distinct(gen_texts, n=2))
            distinct3 = float(_distinct(gen_texts, n=3))

            with open(metrics_path, "w", newline="", encoding="utf-8") as f:
                w = csv.writer(f)
                w.writerow(["fold","final_eval_loss","best_eval_loss","continuity","coherence_jaccard","coherence_semantic","distinct2","distinct3","samples"])
                w.writerow([fold_id, final_eval_loss, best_eval_loss, continuity, coh_j, coh_sem, distinct2, distinct3, len(gen_texts)])

            log(f"[child {fold_id}] metrics: final_eval={final_eval_loss} best_eval={best_eval_loss} cont={continuity:.1f} cohJ={coh_j:.3f} cohSem={coh_sem:.3f} d2={distinct2:.3f} d3={distinct3:.3f}")
            log(f"[child {fold_id}] saved test samples -> {samples_jsonl}")

        except Exception as e:
            with open(metrics_path, "w", newline="", encoding="utf-8") as f:
                w = csv.writer(f); w.writerow(["fold","final_eval_loss","best_eval_loss","continuity","coherence_jaccard","coherence_semantic","distinct2","distinct3","samples"])
            log(f"[child {fold_id}] metrics failed: {e}")

        # cleanup
        del trainer, peft_model, base_model, base_tok, train_ds
        if 'val_ds' in locals(): del val_ds
        if 'train_tok' in locals(): del train_tok
        if 'val_tok' in locals(): del val_tok
        flush_cuda()

        with open(os.path.join(fold_dir,"DONE.json"), "w", encoding="utf-8") as f:
            json.dump({
                "loss_csv": os.path.join(fold_dir, "train_eval_logs.csv"),
                "metrics_csv": metrics_path,
                "samples_jsonl": samples_jsonl,
                "samples_csv": samples_csv
            }, f)
        log(f"========== [child] EXIT run_fold {fold_id} ==========")
        sys.exit(0)

    except Exception as e:
        fold_dir=os.path.join(out_root, f"fold_{fold_id}"); Path(fold_dir).mkdir(parents=True, exist_ok=True)
        with open(os.path.join(fold_dir, "FAIL.log"), "w", encoding="utf-8") as f: traceback.print_exc(file=f)
        with open(os.path.join(fold_dir, "FAIL.json"), "w", encoding="utf-8") as f: json.dump({"error": str(e), "see": "FAIL.log"}, f, indent=2)
        print(f"[{now()}] Fold {fold_id} FAILED. See {os.path.join(fold_dir, 'FAIL.log')}", flush=True)
        flush_cuda()
        sys.exit(1)

if __name__ == "__main__":
    main_worker()
'''

def write_worker(path):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path,"w",encoding="utf-8") as f: f.write(worker_code)

# =========================
# Plot helpers (parent side)
# =========================
def _ensure_dir(p: str):
    Path(p).parent.mkdir(parents=True, exist_ok=True)
    return p

def _plot_per_fold_losses(per_fold_loss_paths, out_root):
    for p in per_fold_loss_paths:
        try:
            df = pd.read_csv(p)
        except Exception:
            continue
        if "phase" not in df.columns or "loss" not in df.columns:
            continue
        fold_id = str(Path(p).parts[-2]).replace("fold_", "")
        dft = df[df["phase"] == "train"].copy()
        if "global_step" in dft.columns and "loss" in dft.columns and len(dft) > 0:
            plt.figure()
            plt.plot(dft["global_step"].values, dft["loss"].astype(float).values)
            plt.xlabel("Global step"); plt.ylabel("Train loss"); plt.title(f"Fold {fold_id}: Train loss")
            out = _ensure_dir(os.path.join(out_root, f"fold_{fold_id}", "plots", "train_loss.png"))
            Path(out).parent.mkdir(parents=True, exist_ok=True)
            plt.savefig(out, dpi=180, bbox_inches="tight"); plt.close()

        dfe = df[df["phase"] == "eval"].copy()
        if "global_step" in dfe.columns and "eval_loss" in dfe.columns and len(dfe) > 0:
            plt.figure()
            x = dfe["global_step"].values
            y = dfe["eval_loss"].astype(float).values
            plt.plot(x, y, marker="o")
            plt.xlabel("Global step"); plt.ylabel("Eval loss"); plt.title(f"Fold {fold_id}: Eval loss")
            out = _ensure_dir(os.path.join(out_root, f"fold_{fold_id}", "plots", "eval_loss.png"))
            Path(out).parent.mkdir(parents=True, exist_ok=True)
            plt.savefig(out, dpi=180, bbox_inches="tight"); plt.close()

def _plot_aggregate_metrics(metrics_csv_out, out_root):
    if not Path(metrics_csv_out).exists():
        return
    df = pd.read_csv(metrics_csv_out)
    if len(df) == 0:
        return
    metrics = ["continuity","coherence_jaccard","coherence_semantic","distinct2","distinct3"]
    means, stds = [], []
    for m in metrics:
        if m in df.columns:
            means.append(float(df[m].mean()))
            stds.append(float(df[m].std(ddof=0)) if len(df)>1 else 0.0)
        else:
            means.append(0.0); stds.append(0.0)

    plt.figure(figsize=(8,4.5))
    x = range(len(metrics))
    plt.bar(x, means, yerr=stds)
    plt.xticks(list(x), metrics, rotation=15)
    plt.title("Aggregate test metrics (mean � std across folds)")
    plt.ylabel("Score")
    out = _ensure_dir(os.path.join(out_root, "aggregate_metrics.png"))
    Path(out).parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(out, dpi=180, bbox_inches="tight"); plt.close()

def _plot_generation_lengths(per_fold_samples_csv_paths, out_root):
    for p in per_fold_samples_csv_paths:
        if not Path(p).exists():
            continue
        try:
            df = pd.read_csv(p)
        except Exception:
            continue
        if "generation" not in df.columns:
            continue
        fold_id = str(Path(p).parts[-2]).replace("fold_", "")
        lengths = df["generation"].fillna("").map(lambda s: len(str(s).split()))
        if len(lengths) == 0:
            continue
        plt.figure()
        plt.hist(lengths.values, bins=20)
        plt.xlabel("Generated length (approx. tokens)"); plt.ylabel("Count")
        plt.title(f"Fold {fold_id}: Generation length histogram")
        out = _ensure_dir(os.path.join(out_root, f"fold_{fold_id}", "plots", "gen_length_hist.png"))
        Path(out).parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(out, dpi=180, bbox_inches="tight"); plt.close()

# =========================
# Parent orchestrator
# =========================
def _stream_subprocess(cmd, env, cwd=None):
    proc = subprocess.Popen(
        cmd, env=env, cwd=cwd,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        bufsize=1, universal_newlines=True
    )
    for line in proc.stdout:
        print(line.rstrip("\n"), flush=True)
    proc.stdout.close()
    return proc.wait()

def main_parent():
    log("=== START main ===")
    pool = merge_train_val_to_pool(TRAIN_JSON, VAL_JSON)
    n = len(pool)
    if n == 0:
        raise ValueError("No examples found in training data.")
    global K_FOLDS
    if n < K_FOLDS:
        log(f"Only {n} examples; reducing K from {K_FOLDS} to {n}.")
        K_FOLDS = n

    folds = kfold_indices(n, k=K_FOLDS, seed=SEED)
    out_root = os.path.join(OUT_DIR, f"kfold_local_{K_FOLDS}_{datetime.now().strftime('%Y%m%d_%H%M%S')}")
    Path(out_root).mkdir(parents=True, exist_ok=True)
    log(f"Output root: {out_root}")

    worker_py = os.path.join(out_root, "_worker_fold.py")
    write_worker(worker_py)

    per_fold_loss = []
    per_fold_metrics = []
    per_fold_sample_csv = []

    base_repo = resolve_base_model()
    log(f"Using base model for PEFT + generation: {base_repo} (PEFT_METHOD={PEFT_METHOD})")

    for k in range(K_FOLDS):
        log(f"\n========== K-FOLD {k+1}/{K_FOLDS} ==========")

        arg = {
            "fold_id": k,
            "base_repo": base_repo,
            "out_root": out_root,
            "test_idx": folds[k],
            "train_json": TRAIN_JSON,
            "val_json": VAL_JSON,
            "seed": SEED + k,
            "STYLE_CARD": STYLE_CARD,
            "ERA_GUIDE": ERA_GUIDE,
            "ERA_TAG": ERA_TAG,
            "config": {
                "PEFT_METHOD": PEFT_METHOD,
                "MAX_LEN": MAX_LEN,
                "EPOCHS": EPOCHS,
                "LR": LR,
                "VALID_FRAC": VALID_FRAC_FROM_TRAIN,
            },
            "ENTITIES_CAP": 40,
            "ENT_PRIORITY": "people,characters,places,orgs,foods,clothing,objects,furniture,rooms,vehicles,weather,digital,other",
        }

        args_path = os.path.join(out_root, f"_args_fold_{k}.json")
        with open(args_path, "w", encoding="utf-8") as f:
            json.dump(arg, f, indent=2)

        env = os.environ.copy()
        env["HF_TOKEN"] = HF_TOKEN or ""
        env["PYTHONUNBUFFERED"] = "1"
        env["TERM"] = "xterm"
        env["FORCE_COLOR"] = "1"
        env["TQDM_DISABLE"] = "0"
        env["TQDM_NOTEBOOK"] = "1"

        try:
            rc = _stream_subprocess([sys.executable, worker_py, args_path], env=env)
        except Exception as e:
            log(f"!! Failed to start worker for fold {k}: {e}", level=1)
            rc = 9

        try:
            if torch.cuda.is_available():
                torch.cuda.synchronize()
                torch.cuda.empty_cache()
                torch.cuda.ipc_collect()
        except Exception:
            pass
        gc.collect()

        fold_dir = os.path.join(out_root, f"fold_{k}")
        done_path = os.path.join(fold_dir, "DONE.json")
        fail_path = os.path.join(fold_dir, "FAIL.json")

        if rc == 0 and os.path.exists(done_path):
            with open(done_path, "r", encoding="utf-8") as f:
                info = json.load(f)
            loss_csv = info.get("loss_csv")
            metrics_csv = info.get("metrics_csv")
            samples_csv = info.get("samples_csv")
            if loss_csv and Path(loss_csv).exists():
                per_fold_loss.append(loss_csv)
                try:
                    df_tail = pd.read_csv(loss_csv).tail(8)
                    print(f"\n[loss table] fold {k}\n{df_tail.to_string(index=False)}")
                except Exception:
                    pass
            if metrics_csv and Path(metrics_csv).exists():
                per_fold_metrics.append(metrics_csv)
            if samples_csv and Path(samples_csv).exists():
                per_fold_sample_csv.append(samples_csv)
        else:
            log(f"!! Fold {k} failed or missing DONE.json.", level=1)
            if os.path.exists(fail_path):
                try:
                    with open(fail_path, "r", encoding="utf-8") as f:
                        fail_info = json.load(f)
                    print(f"[fold {k} FAIL] {fail_info.get('error')}. See FAIL.log", flush=True)
                except Exception:
                    pass

    if per_fold_loss:
        frames = [pd.read_csv(p) for p in per_fold_loss if Path(p).exists()]
        agg = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
        agg_loss_path = os.path.join(out_root, "aggregate_train_eval_logs.csv")
        agg.to_csv(agg_loss_path, index=False)
        log(f"[loss] Aggregate loss CSV: {agg_loss_path}")
        try:
            print("\n[aggregate losses] last 30 rows")
            print(pd.read_csv(agg_loss_path).tail(30).to_string(index=False))
        except Exception:
            pass

    metrics_csv_out = None
    if per_fold_metrics:
        norm_frames = []
        for p in per_fold_metrics:
            df = pd.read_csv(p)
            for c in [
                "final_eval_loss","best_eval_loss","continuity",
                "coherence_jaccard","coherence_semantic","distinct2","distinct3","samples","fold",
            ]:
                if c not in df.columns: df[c] = 0.0
            norm_frames.append(df[["fold","final_eval_loss","best_eval_loss","continuity","coherence_jaccard","coherence_semantic","distinct2","distinct3","samples"]])

        m_agg = pd.concat(norm_frames, ignore_index=True)
        metrics_csv_out = os.path.join(out_root, "aggregate_results.csv")
        m_agg.to_csv(metrics_csv_out, index=False)

        summary = {
            "mean_final_eval_loss": float(m_agg["final_eval_loss"].mean()) if len(m_agg) > 0 else 0.0,
            "mean_best_eval_loss": float(m_agg["best_eval_loss"].mean()) if len(m_agg) > 0 else 0.0,
            "mean_continuity": float(m_agg["continuity"].mean()) if len(m_agg) > 0 else 0.0,
            "mean_coherence_jaccard": float(m_agg["coherence_jaccard"].mean()) if len(m_agg) > 0 else 0.0,
            "mean_coherence_semantic": float(m_agg["coherence_semantic"].mean()) if len(m_agg) > 0 else 0.0,
            "mean_d2": float(m_agg["distinct2"].mean()) if len(m_agg) > 0 else 0.0,
            "mean_d3": float(m_agg["distinct3"].mean()) if len(m_agg) > 0 else 0.0,
        }
        summary["std_final_eval_loss"] = float(m_agg["final_eval_loss"].std(ddof=0)) if len(m_agg) > 1 else 0.0
        summary["std_best_eval_loss"] = float(m_agg["best_eval_loss"].std(ddof=0)) if len(m_agg) > 1 else 0.0

        summary_path = os.path.join(out_root, "aggregate_summary.json")
        with open(summary_path, "w", encoding="utf-8") as f:
            json.dump(summary, f, indent=2)

        print("\n=== Per-fold Metrics ===")
        print(m_agg.sort_values("fold").to_string(index=False))
        print("\n=== Aggregate Summary ===")
        for k, v in summary.items():
            print(f"{k}: {v:.6f}")
        print(f"\nSaved:\n - {metrics_csv_out}\n - {summary_path}")
    else:
        print("\n[WARN] No per-fold metrics files were produced  check child logs.")

    try:
        if per_fold_loss: _plot_per_fold_losses(per_fold_loss, out_root)
        if metrics_csv_out and Path(metrics_csv_out).exists(): _plot_aggregate_metrics(metrics_csv_out, out_root)
        if per_fold_sample_csv: _plot_generation_lengths(per_fold_sample_csv, out_root)
        log("[plots] Saved per-fold loss plots, aggregate metrics plot, and generation length histograms.")
    except Exception as e:
        log(f"[plots] Skipped due to error: {e}", level=1)

    log("=== END main ===")

def try_load_tokenizer(repo_id: str):
    try:
        AutoTokenizer.from_pretrained(repo_id, use_fast=True, token=HF_TOKEN, trust_remote_code=True)
        return True
    except Exception:
        return False

CANDIDATE_MODELS = ["Qwen/Qwen2.5-14B-Instruct"]

def resolve_base_model():
    if BASE_MODEL and try_load_tokenizer(BASE_MODEL): return BASE_MODEL
    for rid in CANDIDATE_MODELS:
        if try_load_tokenizer(rid): return rid
    raise RuntimeError("No working base model found. Provide HF token or set BASE_MODEL.")

def main_guard():
    log("Resolving model & starting")
    base_repo = resolve_base_model()
    log(f"Using base model for NER + PEFT: {base_repo} (PEFT_METHOD={PEFT_METHOD})")
    main_parent()

if __name__ == "__main__" or True:
    main_guard()

[setup] Installing: ['ipython']
[2025-11-03T12:47:05] Resolving model & starting



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


[2025-11-03T12:47:05] Using base model for NER + PEFT: Qwen/Qwen2.5-14B-Instruct (PEFT_METHOD=adalora)
[2025-11-03T12:47:05] === START main ===


Generating train split: 0 examples [00:00, ? examples/s]

[2025-11-03T12:47:06] Pool merged: 171 train + 0 val = 171 total rows
[2025-11-03T12:47:06] Output root: outputs/kfold_local_4_20251103_124706
[2025-11-03T12:47:06] Using base model for PEFT + generation: Qwen/Qwen2.5-14B-Instruct (PEFT_METHOD=adalora)
[2025-11-03T12:47:06] 
========== K-FOLD 1/4 ==========
[2025-11-03T12:47:10] [child 0] test=43 | train+val=128
[2025-11-03T12:47:11] [PEFT] Using 4-bit quantization.
[2025-11-03T12:47:31] [child 0] total_step=64; updates_per_epoch=8
[2025-11-03T12:47:32] [child 0] Starting training (IO: original -> entities+modern).
[2025-11-03T12:47:42] fold=0 TRAIN step=1 epoch=0.13793103448275862 loss=2.1069
{'loss': 2.1069, 'grad_norm': 1.2090944051742554, 'learning_rate': 0.0, 'epoch': 0.14}
[2025-11-03T12:47:51] fold=0 TRAIN step=2 epoch=0.27586206896551724 loss=2.2447
{'loss': 2.2447, 'grad_norm': 1.5318089723587036, 'learning_rate': 9.999999999999999e-06, 'epoch': 0.28}
[2025-11-03T12:48:01] fold=0 TRAIN step=3 epoch=0.41379310344827586 loss=2.0